# GLiNER2-PII — Evaluation Notebook

**Goal:** Evaluate GLiNER2-PII on the same dataset used for SpanMarker, to get a direct apples-to-apples comparison.

This notebook is the **second half** of the AI Privacy Gateway detection layer evaluation:
1. `spanmarker_evaluation.ipynb` — done (F1 = 0.609)
2. **`gliner_evaluation.ipynb`** ← you are here

## Model

`urchade/gliner_multi_pii-v1` — GLiNER fine-tuned on the synthetic PII NER dataset.
- ~0.3B parameters (BERT-like encoder + label-embedding head)
- **Label-conditioned**: you pass entity labels at inference time — no retraining needed
- Character-span output (same as SpanMarker)
- Trained on 50+ PII entity types (person, email, phone, address, passport, IBAN, etc.)
- Apache 2.0 license

## What this notebook measures (same as SpanMarker)

- Strict + partial entity-level F1
- Per-entity-type breakdown
- Latency (median, P95, P99)
- Sample predictions visualization
- Saves metrics to CSV for comparison

## How to use

1. Run all cells top-to-bottom
2. Set `DATASET_PATH` to your `eval_set.jsonl` (same one used for SpanMarker)
3. Compare `gliner_per_type_metrics.csv` with `spanmarker_per_type_metrics.csv` side-by-side


## 1. Install Dependencies

In [1]:
# Install gliner + seqeval (transformers + torch are pre-installed on Kaggle)
!pip install -q gliner seqeval

# Verify
try:
    from gliner import GLiNER
    print("✅ gliner imported successfully")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Falling back to GitHub archive install:")
    !pip install -q https://github.com/urchade/GLiNER/archive/refs/heads/main.zip
    from gliner import GLiNER
    print("✅ gliner imported on retry")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
❌ Import failed: No module named 'gliner'
Falling back to GitHub archive install:
     | 9.9 MB 21.0 MB/s 0:00:000m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ gliner imported on retry


## 2. Imports

In [2]:
import json
import time
import statistics
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple

import torch
from gliner import GLiNER

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


## 3. Configuration

**Important:** GLiNER is label-conditioned — you pass the entity labels you want it to detect at inference time. The labels below match your `eval_set.jsonl` schema.


In [3]:
# Model: GLiNER fine-tuned on synthetic PII dataset
MODEL_NAME = "urchade/gliner_multi_pii-v1"

# Dataset path — use the SAME eval_set.jsonl you used for SpanMarker
DATASET_PATH = "/kaggle/input/datasets/rehabhamdy/ner-eval/eval_set.jsonl"  # ← set to "/kaggle/input/<your-dataset>/eval_set.jsonl"

OUTPUT_DIR = Path("/kaggle/working/") if Path("/kaggle/working/").exists() else Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Confidence threshold for predictions (lower = more recall, less precision)
CONFIDENCE_THRESHOLD = 0.5

N_SAMPLES_TO_DISPLAY = 5

# ─── Entity labels passed to GLiNER at inference ────────────────
# These match the entity types in your eval_set.jsonl.
# GLiNER will only predict entities from this list.
LABELS = [
    # People / orgs / locations
    "person",
    "organization",
    "location",
    "nationality",
    # Contact info
    "email",
    "phone number",
    "username",
    # IDs / financial
    "id number",
    "bank account",
    "credit card",
    "passport number",
    # Digital
    "url",
    "ip address",
    # Time / money
    "date",
    "time",
    "money",
    "percent",
    "ordinal",
    "quantity",
    # Demographics
    "age",
    # General entities (to compare with SpanMarker)
    "work of art",
    "event",
    "product",
    "law",
    "language",
    "facility",
]

print(f"Model: {MODEL_NAME}")
print(f"Labels passed to GLiNER: {len(LABELS)}")
print(f"Dataset: {'Built-in test set' if DATASET_PATH is None else DATASET_PATH}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")


Model: urchade/gliner_multi_pii-v1
Labels passed to GLiNER: 26
Dataset: /kaggle/input/datasets/rehabhamdy/ner-eval/eval_set.jsonl
Confidence threshold: 0.5


## 4. Built-in Test Set (same 12 examples as SpanMarker notebook)

Used when `DATASET_PATH = None` so you can verify the pipeline works.


In [4]:
BUILTIN_TEST_SET = [
    {"text": "Email John Smith at john@acme.com about the Q3 report.",
     "entities": [[6, 16, "PERSON"], [20, 33, "EMAIL"]]},
    {"text": "Sarah Connor from Cyberdyne Systems called about the T-800 prototype.",
     "entities": [[0, 13, "PERSON"], [19, 37, "ORGANIZATION"]]},
    {"text": "The meeting with Apple is scheduled for March 15, 2024 in Cupertino.",
     "entities": [[22, 27, "ORGANIZATION"], [48, 61, "DATE"], [65, 75, "LOCATION"]]},
    {"text": "Please transfer $5,000 to IBAN DE89 3704 0044 0532 0130 00 by Friday.",
     "entities": [[21, 28, "MONEY"], [45, 51, "DATE"]]},
    {"text": "Dr. Maria Rodriguez will present at the Stanford Medical Center on Tuesday.",
     "entities": [[4, 20, "PERSON"], [44, 69, "ORGANIZATION"], [73, 80, "DATE"]]},
    {"text": "Contact support at +1-800-555-1234 or visit https://example.com for help.",
     "entities": [[20, 35, "PHONE_NUMBER"]]},
    {"text": "Liu Wei from Beijing said the project deadline is October 31st.",
     "entities": [[0, 7, "PERSON"], [13, 20, "LOCATION"], [49, 60, "DATE"]]},
    {"text": "The new iPhone 15 was announced at Apple Park in Cupertino, California.",
     "entities": [[8, 17, "PRODUCT"], [41, 51, "ORGANIZATION"], [55, 65, "LOCATION"], [67, 77, "LOCATION"]]},
    {"text": "Albert Einstein published the theory of relativity in 1905 while working in Bern.",
     "entities": [[0, 15, "PERSON"], [67, 71, "DATE"], [89, 93, "LOCATION"]]},
    {"text": "Call Jacques Dupont at +33 1 42 86 82 00 for the Paris office.",
     "entities": [[5, 19, "PERSON"], [23, 41, "PHONE_NUMBER"], [51, 56, "LOCATION"]]},
    {"text": "Amazon Web Services announced a $1.2 billion investment in Dublin on Monday.",
     "entities": [[0, 19, "ORGANIZATION"], [44, 56, "MONEY"], [60, 67, "LOCATION"], [71, 77, "DATE"]]},
    {"text": "Akira Tanaka from Sony will demonstrate the new PlayStation 5 at Tokyo Game Show.",
     "entities": [[0, 12, "PERSON"], [18, 22, "ORGANIZATION"], [58, 71, "PRODUCT"], [75, 90, "EVENT"]]},
]

print(f"Built-in test set: {len(BUILTIN_TEST_SET)} examples")
print(f"Total entities: {sum(len(ex['entities']) for ex in BUILTIN_TEST_SET)}")


Built-in test set: 12 examples
Total entities: 34


## 5. Label Mapping

GLiNER returns labels EXACTLY as you passed them (e.g., `"person"`, `"email"`). We map them back to your `eval_set.jsonl` schema (uppercase: `PERSON`, `EMAIL`).


In [5]:
# Map GLiNER labels (lowercase, space-separated) → eval_set schema (uppercase, underscore)
GLINER_LABEL_MAP = {
    "person": "PERSON",
    "organization": "ORGANIZATION",
    "location": "LOCATION",
    "nationality": "NATIONALITY",
    "email": "EMAIL",
    "phone number": "PHONE_NUMBER",
    "username": "USERNAME",
    "id number": "ID_NUMBER",
    "bank account": "BANK_ACCOUNT",
    "credit card": "CREDIT_CARD",
    "passport number": "PASSPORT_NUMBER",
    "url": "URL",
    "ip address": "IP_ADDRESS",
    "date": "DATE",
    "time": "TIME",
    "money": "MONEY",
    "percent": "PERCENT",
    "ordinal": "ORDINAL",
    "quantity": "QUANTITY",
    "age": "AGE",
    "work of art": "WORK_OF_ART",
    "event": "EVENT",
    "product": "PRODUCT",
    "law": "LAW",
    "language": "LANGUAGE",
    "facility": "FAC",
}

# Build the inverse map for verification
EVAL_LABELS = set(GLINER_LABEL_MAP.values())
print(f"GLiNER will predict {len(LABELS)} entity types")
print(f"Mapped to {len(EVAL_LABELS)} eval schema labels")


GLiNER will predict 26 entity types
Mapped to 26 eval schema labels


## 6. Load Dataset

In [6]:
def load_dataset(path):
    if path is None:
        print("ℹ️  DATASET_PATH is None — using built-in test set")
        return BUILTIN_TEST_SET
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try:
                ex = json.loads(line)
                assert "text" in ex and isinstance(ex["text"], str)
                assert "entities" in ex and isinstance(ex["entities"], list)
                for e in ex["entities"]:
                    assert len(e) == 3
                    assert isinstance(e[0], int) and isinstance(e[1], int)
                    assert isinstance(e[2], str)
                    assert 0 <= e[0] < e[1] <= len(ex["text"])
                examples.append(ex)
            except Exception as e:
                print(f"⚠️  Skipping line {i}: {e}")
    print(f"✅ Loaded {len(examples)} examples from {path}")
    print(f"   Total entities: {sum(len(ex['entities']) for ex in examples)}")
    return examples

DATASET = load_dataset(DATASET_PATH)

# Stats
ent_types = defaultdict(int)
for ex in DATASET:
    for _, _, label in ex["entities"]:
        ent_types[label] += 1
print(f"\nEntity type distribution:")
for label, count in sorted(ent_types.items(), key=lambda x: -x[1]):
    print(f"  {label:25s} {count:4d}")


✅ Loaded 80 examples from /kaggle/input/datasets/rehabhamdy/ner-eval/eval_set.jsonl
   Total entities: 278

Entity type distribution:
  PERSON                      64
  ORGANIZATION                53
  DATE                        33
  LOCATION                    33
  MONEY                       19
  EMAIL                        9
  ID_NUMBER                    6
  TIME                         6
  USERNAME                     6
  IP_ADDRESS                   6
  PRODUCT                      5
  PERCENT                      4
  URL                          4
  LANGUAGE                     4
  FAC                          4
  LAW                          4
  NATIONALITY                  3
  EVENT                        3
  ORDINAL                      2
  WORK_OF_ART                  2
  PHONE_NUMBER                 2
  BANK_ACCOUNT                 2
  QUANTITY                     1
  AGE                          1
  CREDIT_CARD                  1
  PASSPORT_NUMBER              1


## 7. Load GLiNER Model

In [7]:
print(f"Loading model: {MODEL_NAME}")
print("⏳ This may take 30-60 seconds on first run (downloads ~440 MB)...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

t0 = time.time()
model = GLiNER.from_pretrained(MODEL_NAME)
model.eval()
# Move to device (GLiNER uses .to() like a torch model)
if device == "cuda":
    model = model.to(device)
print(f"✅ Model loaded in {time.time() - t0:.1f}s")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")


Loading model: urchade/gliner_multi_pii-v1
⏳ This may take 30-60 seconds on first run (downloads ~440 MB)...
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

✅ Model loaded in 19.8s
   Parameters: 288.9M


## 8. Run Inference

GLiNER API:
```python
entities = model.predict_entities(text, labels, threshold=0.5)
# Returns: [{"text": "...", "label": "person", "start": 0, "end": 4, "score": 0.95}, ...]
```

We map each prediction's label back to our eval schema and apply the confidence threshold.


In [8]:
def normalize_pred(p, text, threshold=0.5):
    """Convert a GLiNER prediction to {start, end, label, score, text}."""
    if not isinstance(p, dict):
        return None
    start = p.get("start")
    end = p.get("end")
    raw_label = p.get("label")
    score = p.get("score", 0)
    if start is None or end is None or raw_label is None:
        return None
    if score < threshold:
        return None
    mapped = GLINER_LABEL_MAP.get(raw_label)
    if mapped is None:
        return None
    try:
        start = int(start)
        end = int(end)
    except (TypeError, ValueError):
        return None
    if not (0 <= start < end <= len(text)):
        return None
    return {
        "start": start,
        "end": end,
        "label": mapped,
        "score": float(score),
        "text": text[start:end],
    }


def run_inference(model, examples, labels, threshold=0.5):
    all_preds = []
    latencies = []
    unmapped = set()
    for i, ex in enumerate(examples):
        text = ex["text"]
        t0 = time.time()
        with torch.no_grad():
            raw = model.predict_entities(text, labels, threshold=threshold)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        preds = []
        for p in raw:
            raw_label = p.get("label", "")
            if raw_label and raw_label not in GLINER_LABEL_MAP:
                unmapped.add(raw_label)
            n = normalize_pred(p, text, threshold)
            if n is not None:
                preds.append(n)
        all_preds.append(preds)

        if (i + 1) % 25 == 0 or i == len(examples) - 1:
            print(f"  [{i+1}/{len(examples)}]  last={latency_ms:.1f}ms  preds={len(preds)}")

    if unmapped:
        print(f"\n⚠️  Unmapped GLiNER labels (skipped): {sorted(unmapped)}")
    return all_preds, latencies


print(f"Running inference on {len(DATASET)} examples...")
print(f"Passing {len(LABELS)} labels to GLiNER at inference time...")
t0 = time.time()
PREDICTIONS, LATENCIES = run_inference(model, DATASET, LABELS, CONFIDENCE_THRESHOLD)
total_s = time.time() - t0
print(f"\n✅ Done in {total_s:.1f}s ({total_s/len(DATASET)*1000:.1f}ms/example avg)")
print(f"   Total predictions: {sum(len(p) for p in PREDICTIONS)}")
print(f"   Total gold entities: {sum(len(ex['entities']) for ex in DATASET)}")


Running inference on 80 examples...
Passing 26 labels to GLiNER at inference time...
  [25/80]  last=29.6ms  preds=3
  [50/80]  last=28.0ms  preds=3
  [75/80]  last=28.7ms  preds=3
  [80/80]  last=29.0ms  preds=7

✅ Done in 4.3s (53.3ms/example avg)
   Total predictions: 303
   Total gold entities: 278


## 9. Latency Benchmark

In [9]:
lat_sorted = sorted(LATENCIES)
n = len(lat_sorted)
median = statistics.median(lat_sorted)
p95 = lat_sorted[int(n * 0.95)] if n >= 20 else max(lat_sorted)
p99 = lat_sorted[int(n * 0.99)] if n >= 100 else max(lat_sorted)

print(f"Latency statistics ({n} examples, device={device}):")
print(f"  Median:  {median:7.1f} ms")
print(f"  P95:     {p95:7.1f} ms")
print(f"  P99:     {p99:7.1f} ms")
print(f"  Min:     {min(lat_sorted):7.1f} ms")
print(f"  Max:     {max(lat_sorted):7.1f} ms")
print(f"  Mean:    {sum(lat_sorted)/n:7.1f} ms")
print()
print(f"vs. project budget: P95 ≤ 400ms")
print(f"  → GLiNER alone uses {p95/400*100:.0f}% of the P95 budget")
print(f"  → SpanMarker was 76ms P95 ({76/400*100:.0f}% of budget)")
print(f"  → Combined SpanMarker + GLiNER: ~{76 + p95:.0f}ms P95 ({(76 + p95)/400*100:.0f}% of budget)")


Latency statistics (80 examples, device=cuda):
  Median:     29.6 ms
  P95:       155.1 ms
  P99:       979.9 ms
  Min:        27.8 ms
  Max:       979.9 ms
  Mean:       53.3 ms

vs. project budget: P95 ≤ 400ms
  → GLiNER alone uses 39% of the P95 budget
  → SpanMarker was 76ms P95 (19% of budget)
  → Combined SpanMarker + GLiNER: ~231ms P95 (58% of budget)


## 10. Entity-Level Metrics

Same scoring as SpanMarker notebook (strict + partial), so the two CSVs are directly comparable.


In [10]:
def spans_overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]

def compute_metrics(examples, predictions, mode="strict"):
    tp = fp = fn = 0
    for ex, preds in zip(examples, predictions):
        gold = [(e[0], e[1], e[2]) for e in ex["entities"]]
        pred = [(p["start"], p["end"], p["label"]) for p in preds]
        matched_gold = set()
        for i, p in enumerate(pred):
            for j, g in enumerate(gold):
                if j in matched_gold or p[2] != g[2]: continue
                if mode == "strict":
                    if p[0] == g[0] and p[1] == g[1]:
                        tp += 1; matched_gold.add(j); break
                else:
                    if spans_overlap((p[0], p[1]), (g[0], g[1])):
                        tp += 1; matched_gold.add(j); break
            else:
                fp += 1
        fn += len(gold) - len(matched_gold)
    p = tp/(tp+fp) if (tp+fp)>0 else 0.0
    r = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1 = 2*p*r/(p+r) if (p+r)>0 else 0.0
    return {"mode": mode, "tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": f1}

metrics_strict = compute_metrics(DATASET, PREDICTIONS, "strict")
metrics_partial = compute_metrics(DATASET, PREDICTIONS, "partial")

print("╔══════════════════════════════════════════════════════════╗")
print("║              ENTITY-LEVEL METRICS                       ║")
print("╠══════════════════════╦════════════╦════════════╦═══════╣")
print("║ Metric               ║  Strict    ║  Partial   ║       ║")
print("╠══════════════════════╬════════════╬════════════╬═══════╣")
print(f"║ Precision            ║  {metrics_strict['precision']:.4f}   ║  {metrics_partial['precision']:.4f}    ║       ║")
print(f"║ Recall               ║  {metrics_strict['recall']:.4f}   ║  {metrics_partial['recall']:.4f}    ║       ║")
print(f"║ F1                   ║  {metrics_strict['f1']:.4f}   ║  {metrics_partial['f1']:.4f}    ║       ║")
print(f"║ TP                   ║  {metrics_strict['tp']:>5}     ║  {metrics_partial['tp']:>5}      ║       ║")
print(f"║ FP                   ║  {metrics_strict['fp']:>5}     ║  {metrics_partial['fp']:>5}      ║       ║")
print(f"║ FN                   ║  {metrics_strict['fn']:>5}     ║  {metrics_partial['fn']:>5}      ║       ║")
print("╚══════════════════════╩════════════╩════════════╩═══════╝")
print()
print(f"📊 For comparison: SpanMarker strict F1 was 0.6095")
print(f"                  SpanMarker partial F1 was 0.6862")


╔══════════════════════════════════════════════════════════╗
║              ENTITY-LEVEL METRICS                       ║
╠══════════════════════╦════════════╦════════════╦═══════╣
║ Metric               ║  Strict    ║  Partial   ║       ║
╠══════════════════════╬════════════╬════════════╬═══════╣
║ Precision            ║  0.7525   ║  0.8053    ║       ║
║ Recall               ║  0.8201   ║  0.8777    ║       ║
║ F1                   ║  0.7849   ║  0.8399    ║       ║
║ TP                   ║    228     ║    244      ║       ║
║ FP                   ║     75     ║     59      ║       ║
║ FN                   ║     50     ║     34      ║       ║
╚══════════════════════╩════════════╩════════════╩═══════╝

📊 For comparison: SpanMarker strict F1 was 0.6095
                  SpanMarker partial F1 was 0.6862


## 11. Per-Entity-Type Breakdown

In [11]:
def compute_per_type_metrics(examples, predictions, mode="strict"):
    per_type = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    for ex, preds in zip(examples, predictions):
        gold = [(e[0], e[1], e[2]) for e in ex["entities"]]
        pred = [(p["start"], p["end"], p["label"]) for p in preds]
        matched_gold = set()
        for p_start, p_end, p_label in pred:
            for j, (g_start, g_end, g_label) in enumerate(gold):
                if j in matched_gold or g_label != p_label: continue
                if mode == "strict":
                    if p_start == g_start and p_end == g_end:
                        per_type[p_label]["tp"] += 1; matched_gold.add(j); break
                else:
                    if spans_overlap((p_start, p_end), (g_start, g_end)):
                        per_type[p_label]["tp"] += 1; matched_gold.add(j); break
            else:
                per_type[p_label]["fp"] += 1
        for j, (_, _, g_label) in enumerate(gold):
            if j not in matched_gold:
                per_type[g_label]["fn"] += 1
    rows = []
    for label, counts in per_type.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        p = tp/(tp+fp) if (tp+fp)>0 else 0.0
        r = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1 = 2*p*r/(p+r) if (p+r)>0 else 0.0
        rows.append({"entity_type": label, "tp": tp, "fp": fp, "fn": fn,
                     "support": tp+fn, "precision": p, "recall": r, "f1": f1})
    rows.sort(key=lambda r: -r["support"])
    return rows

per_type_strict = compute_per_type_metrics(DATASET, PREDICTIONS, "strict")

print(f"\nPer-Entity-Type (STRICT mode):")
print(f"{'Entity Type':<25} {'Support':>8} {'TP':>5} {'FP':>5} {'FN':>5} {'P':>7} {'R':>7} {'F1':>7}")
print("─" * 78)
for r in per_type_strict:
    print(f"{r['entity_type']:<25} {r['support']:>8} {r['tp']:>5} {r['fp']:>5} {r['fn']:>5} "
          f"{r['precision']:>7.3f} {r['recall']:>7.3f} {r['f1']:>7.3f}")
print("─" * 78)
total_support = sum(r["support"] for r in per_type_strict)
total_tp = sum(r["tp"] for r in per_type_strict)
total_fp = sum(r["fp"] for r in per_type_strict)
total_fn = sum(r["fn"] for r in per_type_strict)
p = total_tp/(total_tp+total_fp); r = total_tp/(total_tp+total_fn)
f1 = 2*p*r/(p+r) if (p+r)>0 else 0
print(f"{'TOTAL':<25} {total_support:>8} {total_tp:>5} {total_fp:>5} {total_fn:>5} {p:>7.3f} {r:>7.3f} {f1:>7.3f}")



Per-Entity-Type (STRICT mode):
Entity Type                Support    TP    FP    FN       P       R      F1
──────────────────────────────────────────────────────────────────────────────
PERSON                          64    52    15    12   0.776   0.812   0.794
ORGANIZATION                    53    48     9     5   0.842   0.906   0.873
DATE                            33    29     1     4   0.967   0.879   0.921
LOCATION                        33    25     7     8   0.781   0.758   0.769
MONEY                           19    19     2     0   0.905   1.000   0.950
EMAIL                            9     9     6     0   0.600   1.000   0.750
ID_NUMBER                        6     1     0     5   1.000   0.167   0.286
TIME                             6     6     7     0   0.462   1.000   0.632
USERNAME                         6     5     0     1   1.000   0.833   0.909
IP_ADDRESS                       6     2     0     4   1.000   0.333   0.500
PRODUCT                          5     5  

## 12. Sample Predictions vs Ground Truth

Green = TP, Red = FN (missed), Yellow = FP (invented).


In [14]:
from IPython.display import HTML, display

COLORS = {
    "tp": "#10B981",  # green
    "fn": "#EF4444",  # red
    "fp": "#F59E0B",  # amber
}


def render_example_html(text, gold, pred):
    """Render an example with colored TP/FN/FP entity highlights."""

    # Convert gold entities to tuples so they can be used in sets
    gold_tuples = [
        (g[0], g[1], g[2])
        for g in gold
    ]

    gold_set = set(gold_tuples)

    pred_set = set(
        (p["start"], p["end"], p["label"])
        for p in pred
    )

    spans_to_render = []

    # ---------------------------------------------------------
    # GOLD ENTITIES
    # ---------------------------------------------------------
    for g_start, g_end, g_label in gold_tuples:

        # Exact match
        in_pred = (
            g_start,
            g_end,
            g_label
        ) in pred_set

        # Partial overlap with same label
        if not in_pred:
            for p in pred:
                if (
                    p["label"] == g_label
                    and spans_overlap(
                        (p["start"], p["end"]),
                        (g_start, g_end)
                    )
                ):
                    in_pred = True
                    break

        spans_to_render.append(
            (
                g_start,
                g_end,
                "tp" if in_pred else "fn",
                g_label
            )
        )

    # ---------------------------------------------------------
    # PREDICTED ENTITIES
    # ---------------------------------------------------------
    for p in pred:

        pred_tuple = (
            p["start"],
            p["end"],
            p["label"]
        )

        # Exact match
        in_gold = pred_tuple in gold_set

        # Partial overlap with same label
        if not in_gold:
            for g_start, g_end, g_label in gold_tuples:
                if (
                    g_label == p["label"]
                    and spans_overlap(
                        (g_start, g_end),
                        (p["start"], p["end"])
                    )
                ):
                    in_gold = True
                    break

        # Only unmatched predictions are FP
        if not in_gold:
            spans_to_render.append(
                (
                    p["start"],
                    p["end"],
                    "fp",
                    p["label"]
                )
            )

    # ---------------------------------------------------------
    # SORT SPANS
    # ---------------------------------------------------------
    spans_to_render.sort(
        key=lambda s: (s[0], s[1])
    )

    # ---------------------------------------------------------
    # BUILD HTML
    # ---------------------------------------------------------
    html_parts = []
    last_end = 0

    for start, end, kind, label in spans_to_render:

        # Prevent overlapping HTML spans
        if start < last_end:
            start = last_end

        # Add normal text before the entity
        if start > last_end:
            html_parts.append(
                f"<span>{text[last_end:start]}</span>"
            )

        color = COLORS[kind]

        html_parts.append(
            f'<span style="'
            f'background-color: {color}33; '
            f'border-bottom: 2px solid {color}; '
            f'padding: 1px 2px;" '
            f'title="{kind.upper()} - {label}">'
            f'{text[start:end]}'
            f'<sub style="font-size:9px;color:{color}">'
            f' [{label}]'
            f'</sub>'
            f'</span>'
        )

        last_end = end

    # Add remaining text
    if last_end < len(text):
        html_parts.append(
            f"<span>{text[last_end:]}</span>"
        )

    return "".join(html_parts)


# =============================================================
# SHOW SAMPLE PREDICTIONS
# =============================================================
N_SAMPLES_TO_DISPLAY = 20
print(
    f"\nShowing {N_SAMPLES_TO_DISPLAY} sample predictions:\n"
)

for i, (ex, preds) in enumerate(
    zip(
        DATASET[:N_SAMPLES_TO_DISPLAY],
        PREDICTIONS[:N_SAMPLES_TO_DISPLAY]
    )
):

    gold = ex["entities"]

    html = render_example_html(
        ex["text"],
        gold,
        preds
    )

    print(f"─── Example {i + 1} ───")
    print(f"Text: {ex['text']}")
    print(f"Gold entities: {gold}")

    prediction_summary = [
        (
            p["start"],
            p["end"],
            p["label"],
            round(p["score"], 2)
        )
        for p in preds
    ]

    print(f"Predictions:  {prediction_summary}")
    print()

    display(
        HTML(
            f'''
            <div style="
                font-family: monospace;
                padding: 8px;
                background: #F8FAFC;
                border-radius: 4px;
            ">
                {html}
            </div>
            '''
        )
    )

    print()


Showing 20 sample predictions:

─── Example 1 ───
Text: Hi Maria Rodriguez, please confirm the Cyberdyne Systems contract by March 15, 2025. Reply to maria.rodriguez@cyberdyne.com.
Gold entities: [[3, 18, 'PERSON'], [39, 56, 'ORGANIZATION'], [69, 83, 'DATE'], [94, 123, 'EMAIL']]
Predictions:  [(3, 18, 'PERSON', 1.0), (39, 56, 'ORGANIZATION', 1.0), (69, 83, 'DATE', 1.0), (94, 123, 'EMAIL', 0.98)]




─── Example 2 ───
Text: Draft an email to Akira Tanaka at Toyota thanking him for the meeting on Monday.
Gold entities: [[18, 30, 'PERSON'], [34, 40, 'ORGANIZATION'], [73, 79, 'DATE']]
Predictions:  [(9, 14, 'EMAIL', 0.92), (18, 30, 'PERSON', 1.0), (34, 40, 'ORGANIZATION', 1.0), (62, 69, 'EVENT', 0.9), (73, 79, 'DATE', 0.99)]




─── Example 3 ───
Text: Subject: Invoice follow-up. Dear Mr. Liu Wei, we have not received payment for invoice INV-20431 from Huawei Technologies.
Gold entities: [[37, 44, 'PERSON'], [87, 96, 'ID_NUMBER'], [102, 121, 'ORGANIZATION']]
Predictions:  [(9, 26, 'EVENT', 0.64), (33, 44, 'PERSON', 0.99), (79, 96, 'MONEY', 0.56), (102, 121, 'ORGANIZATION', 1.0)]




─── Example 4 ───
Text: Write a polite reminder to jacques.dupont@airfrance.fr that his flight booking expires on 12/05/2025.
Gold entities: [[27, 54, 'EMAIL'], [90, 100, 'DATE']]
Predictions:  [(27, 54, 'EMAIL', 0.96), (64, 78, 'EVENT', 0.6), (90, 100, 'DATE', 1.0)]




─── Example 5 ───
Text: Hi Priya, can you forward Sven's slides to sven.andersson@volvo.se before Thursday?
Gold entities: [[3, 8, 'PERSON'], [26, 30, 'PERSON'], [43, 66, 'EMAIL'], [74, 82, 'DATE']]
Predictions:  [(3, 8, 'PERSON', 0.99), (43, 66, 'EMAIL', 0.98), (74, 82, 'DATE', 0.94)]



...


─── Example 16 ───
Text: Write a follow-up to Yuki Sato: thanks for the demo, and can we schedule the second call next week?
Gold entities: [[21, 30, 'PERSON'], [77, 83, 'ORDINAL'], [89, 98, 'DATE']]
Predictions:  [(21, 30, 'PERSON', 1.0), (47, 51, 'EVENT', 0.74), (89, 98, 'DATE', 0.87)]




─── Example 17 ───
Text: Reminder: send the quarterly summary to finance@acme-corp.com by end of day.
Gold entities: [[40, 61, 'EMAIL']]
Predictions:  [(19, 36, 'MONEY', 0.56), (40, 61, 'EMAIL', 0.93), (65, 75, 'TIME', 0.82)]




─── Example 18 ───
Text: Please write a short thank-you email to the hiring manager.
Gold entities: []
Predictions:  [(44, 58, 'PERSON', 0.86)]




─── Example 19 ───
Text: Dear Ms. Okafor, your appointment at St. Mary's Hospital is confirmed for 9:30 AM on August 8.
Gold entities: [[9, 15, 'PERSON'], [37, 56, 'ORGANIZATION'], [74, 81, 'TIME'], [85, 93, 'DATE']]
Predictions:  [(5, 15, 'PERSON', 1.0), (22, 33, 'EVENT', 0.54), (37, 56, 'FAC', 0.81), (74, 81, 'TIME', 0.99), (85, 93, 'DATE', 0.99)]




─── Example 20 ───
Text: Yes, please email Maria the contract for the Q4 review.
Gold entities: [[18, 23, 'PERSON'], [45, 47, 'DATE']]
Predictions:  [(12, 17, 'EMAIL', 0.9), (18, 23, 'PERSON', 0.99), (45, 54, 'EVENT', 0.92)]



## 13. Save Metrics

Saves CSV files you can download from Kaggle and compare side-by-side with SpanMarker's CSVs.


In [15]:
import csv

overall_path = OUTPUT_DIR / "gliner_overall_metrics.csv"
with open(overall_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "strict", "partial"])
    w.writerow(["precision", metrics_strict["precision"], metrics_partial["precision"]])
    w.writerow(["recall", metrics_strict["recall"], metrics_partial["recall"]])
    w.writerow(["f1", metrics_strict["f1"], metrics_partial["f1"]])
    w.writerow(["tp", metrics_strict["tp"], metrics_partial["tp"]])
    w.writerow(["fp", metrics_strict["fp"], metrics_partial["fp"]])
    w.writerow(["fn", metrics_strict["fn"], metrics_partial["fn"]])
    w.writerow(["n_examples", len(DATASET), len(DATASET)])
    w.writerow(["median_latency_ms", median, median])
    w.writerow(["p95_latency_ms", p95, p95])
    w.writerow(["device", device, device])
    w.writerow(["model", MODEL_NAME, MODEL_NAME])
    w.writerow(["confidence_threshold", CONFIDENCE_THRESHOLD, CONFIDENCE_THRESHOLD])
print(f"✅ Saved overall metrics: {overall_path}")

per_type_path = OUTPUT_DIR / "gliner_per_type_metrics.csv"
with open(per_type_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["entity_type", "support", "tp", "fp", "fn", "precision", "recall", "f1"])
    for r in per_type_strict:
        w.writerow([r["entity_type"], r["support"], r["tp"], r["fp"], r["fn"],
                    r["precision"], r["recall"], r["f1"]])
print(f"✅ Saved per-type metrics: {per_type_path}")

raw_path = OUTPUT_DIR / "gliner_raw_predictions.jsonl"
with open(raw_path, "w") as f:
    for ex, preds in zip(DATASET, PREDICTIONS):
        out = {"text": ex["text"], "gold": ex["entities"],
               "pred": [[p["start"], p["end"], p["label"], round(p["score"], 4)] for p in preds]}
        f.write(json.dumps(out) + "\n")
print(f"✅ Saved raw predictions: {raw_path}")

print(f"\n{'='*60}")
print(f"SUMMARY (vs SpanMarker)")
print(f"{'='*60}")
print(f"Model:          {MODEL_NAME}")
print(f"Examples:       {len(DATASET)}")
print(f"Device:         {device}")
print(f"Strict F1:      {metrics_strict['f1']:.4f}   (SpanMarker: 0.6095)")
print(f"Partial F1:     {metrics_partial['f1']:.4f}   (SpanMarker: 0.6862)")
print(f"Strict P/R:     {metrics_strict['precision']:.4f} / {metrics_strict['recall']:.4f}")
print(f"                (SpanMarker: 0.8182 / 0.4856)")
print(f"Median latency: {median:.1f} ms  (SpanMarker: 48.6 ms)")
print(f"P95 latency:    {p95:.1f} ms  (SpanMarker: 76.2 ms)")
print(f"\n👉 Compare per_type CSVs side-by-side to decide merge layer strategy")


✅ Saved overall metrics: /kaggle/working/gliner_overall_metrics.csv
✅ Saved per-type metrics: /kaggle/working/gliner_per_type_metrics.csv
✅ Saved raw predictions: /kaggle/working/gliner_raw_predictions.jsonl

SUMMARY (vs SpanMarker)
Model:          urchade/gliner_multi_pii-v1
Examples:       80
Device:         cuda
Strict F1:      0.7849   (SpanMarker: 0.6095)
Partial F1:     0.8399   (SpanMarker: 0.6862)
Strict P/R:     0.7525 / 0.8201
                (SpanMarker: 0.8182 / 0.4856)
Median latency: 29.6 ms  (SpanMarker: 48.6 ms)
P95 latency:    155.1 ms  (SpanMarker: 76.2 ms)

👉 Compare per_type CSVs side-by-side to decide merge layer strategy


## 14. Next Steps

After this notebook finishes, you'll have:

| File | From | Purpose |
|---|---|---|
| `spanmarker_per_type_metrics.csv` | Notebook 1 | SpanMarker baseline |
| `gliner_per_type_metrics.csv` | This notebook | GLiNER baseline |
| `spanmarker_overall_metrics.csv` | Notebook 1 | Overall numbers |
| `gliner_overall_metrics.csv` | This notebook | Overall numbers |

### How to compare

Open both per-type CSVs side-by-side. For each entity type, note:
- **Which model wins** (higher F1)
- **By how much** (10+ point difference is significant)
- **Tradeoffs** (e.g., SpanMarker might have higher precision but lower recall on ORG)

### Expected outcomes (based on SpanMarker's gaps)

GLiNER2-PII should significantly outperform SpanMarker on:
- **EMAIL, PHONE, URL, IP_ADDRESS** (SpanMarker = 0, GLiNER trained on these)
- **DATE, TIME, MONEY** (SpanMarker can't predict these; GLiNER can)
- **ID_NUMBER, USERNAME, BANK_ACCOUNT, PASSPORT_NUMBER, CREDIT_CARD** (PII-specific)

GLiNER may be weaker on:
- **PERSON, ORGANIZATION, LOCATION** (SpanMarker's FewNERD training is broader here)
- **LANGUAGE, FAC, WORK_OF_ART** (not PII-specific, GLiNER's training doesn't emphasize these)

### What to do with the comparison

Once you have both CSVs, share them back with me and I'll help you build:
1. The **merge layer** that takes the best of both per entity type
2. The **routing table** that decides which model to call first
3. The **dedup layer** that resolves conflicts when both models predict the same span with different labels
